# Production Risk Pipeline — PSI Monitor + LLM Triage

Two things a production fraud model needs beyond good metrics:

1. **Drift detection** — the model was trained on historical data. If live transaction patterns shift, scores degrade silently. PSI (Population Stability Index) catches this before it becomes a business problem.
2. **Case writeups** — investigators can't read model scores. They need to know *why* a transaction was flagged, in plain English. This notebook generates SHAP-backed SAR narratives automatically.

Run `02_model_training_validation.ipynb` first.

**LLM narratives require an OpenRouter API key** — free at [openrouter.ai/keys](https://openrouter.ai/keys). Without it the notebook falls back to template narratives.

In [ ]:
import os
os.environ.setdefault('NUMBA_DISABLE_CUDA', '1')

try:
    from dotenv import load_dotenv
    _HERE = os.path.dirname(os.path.abspath('__file__'))
    load_dotenv(os.path.join(_HERE, '..', '..', '.env'))
except ImportError:
    _HERE = os.path.dirname(os.path.abspath('__file__'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, warnings
warnings.filterwarnings('ignore')
import shap

OUT_DIR   = os.path.join(_HERE, '..', 'outputs') + os.sep
MODEL_DIR = os.path.join(_HERE, '..', 'models') + os.sep

In [ ]:
X_train   = pd.read_parquet(OUT_DIR + 'X_train.parquet')
y_train   = pd.read_parquet(OUT_DIR + 'y_train.parquet')['isFraud']
X_test    = pd.read_parquet(OUT_DIR + 'X_test.parquet')
xgb_model = joblib.load(MODEL_DIR + 'challenger_xgboost.pkl')

with open(OUT_DIR + 'features.json') as f:
    FEATURES = json.load(f)

print(f"Model loaded  |  {len(FEATURES)} features")

## PSI Drift Monitor

PSI compares score distributions across 10 bins between a reference population (training) and an actual population (test/live). It's the standard SR 11-7 metric for ongoing model monitoring — in production this would run nightly.

| PSI | Status |
|-----|--------|
| < 0.10 | Stable |
| 0.10 – 0.25 | Minor drift — monitor |
| > 0.25 | Major drift — model review required |

In [ ]:
def compute_psi(expected, actual, n_bins=10):
    expected    = np.array(expected)
    actual      = np.array(actual)
    breakpoints = np.linspace(0, 1, n_bins + 1)

    expected_pcts, actual_pcts = [], []
    for i in range(n_bins):
        lo, hi = breakpoints[i], breakpoints[i+1]
        expected_pcts.append(max(np.mean((expected >= lo) & (expected < hi)), 1e-4))
        actual_pcts.append(max(np.mean((actual   >= lo) & (actual   < hi)), 1e-4))

    expected_pcts = np.array(expected_pcts)
    actual_pcts   = np.array(actual_pcts)
    psi_bins      = (actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts)

    psi_df = pd.DataFrame({
        'bin':              [f'{breakpoints[i]:.1f}–{breakpoints[i+1]:.1f}' for i in range(n_bins)],
        'expected_pct':     (expected_pcts * 100).round(2),
        'actual_pct':       (actual_pcts   * 100).round(2),
        'psi_contribution': psi_bins.round(5)
    })
    return round(psi_bins.sum(), 5), psi_df


def interpret_psi(psi):
    if psi < 0.10:  return 'STABLE'
    if psi < 0.25:  return 'MINOR DRIFT — monitor closely'
    return 'MAJOR DRIFT — model review required'

In [ ]:
train_scores = xgb_model.predict_proba(X_train)[:, 1]
test_scores  = xgb_model.predict_proba(X_test)[:, 1]

psi_value, psi_table = compute_psi(train_scores, test_scores)

print(f"PSI (train vs test): {psi_value:.5f}")
print(f"Assessment         : {interpret_psi(psi_value)}")
print()
print(psi_table.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'PSI Drift Monitor — PSI = {psi_value:.4f}  ({interpret_psi(psi_value)})', fontsize=11)

axes[0].hist(train_scores, bins=50, alpha=0.6, color='#1D9E75', label='Training', density=True)
axes[0].hist(test_scores,  bins=50, alpha=0.6, color='#534AB7', label='Test',     density=True)
axes[0].set_xlabel('Fraud probability score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score distributions')
axes[0].legend()

colors = ['#D85A30' if v > 0.025 else '#1D9E75' for v in psi_table['psi_contribution']]
axes[1].bar(psi_table['bin'], psi_table['psi_contribution'], color=colors)
axes[1].axhline(y=0.01, color='orange', linestyle='--', linewidth=1)
axes[1].set_xlabel('Score bin')
axes[1].set_ylabel('PSI contribution')
axes[1].set_title('PSI per score bin')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUT_DIR + '06_psi_monitor.png', dpi=150, bbox_inches='tight')
plt.show()

## LLM Triage Agent

The model flags ~3,000 transactions a day. Investigators can realistically review ~200. The bottleneck isn't scoring — it's writing up each case.

This layer takes a flagged transaction, computes SHAP values to see which features drove the score, and passes those to an LLM to write a SAR-style case narrative. What used to take 15 minutes of analyst time per case becomes a one-second API call.

The prompt is structured so the LLM acts as a financial crime analyst, not a general assistant. It gets the fraud score, the top risk factors with their SHAP contributions, and writes for a compliance officer audience.

In [ ]:
print("Setting up SHAP explainer...")
explainer = shap.TreeExplainer(xgb_model)
print("Ready")

In [ ]:
def explain_transaction(transaction_row, model, explainer, feature_names, top_n=5):
    row_df = pd.DataFrame([transaction_row], columns=feature_names)
    score  = model.predict_proba(row_df)[0, 1]
    sv     = explainer.shap_values(row_df)[0]

    contrib = pd.DataFrame({
        'feature': feature_names,
        'value':   transaction_row,
        'shap':    sv
    }).sort_values('shap', ascending=False)

    return score, contrib.head(top_n), contrib.tail(top_n).sort_values('shap')

In [ ]:
def _risk_level(score):
    if score < 0.30: return 'LOW'
    if score < 0.50: return 'MEDIUM'
    if score < 0.75: return 'HIGH'
    return 'CRITICAL'


def _template_narrative(score, top_risk):
    level   = _risk_level(score)
    factor1 = top_risk.iloc[0]['feature'] if len(top_risk) > 0 else 'unknown'
    factor2 = top_risk.iloc[1]['feature'] if len(top_risk) > 1 else 'unknown'
    actions = {
        'LOW':      'Auto-approve transaction.',
        'MEDIUM':   'Route to manual review queue within 4 hours.',
        'HIGH':     'Escalate to senior analyst. Consider temporary hold.',
        'CRITICAL': 'Immediate block recommended. Notify compliance officer.'
    }
    return (
        f"Transaction flagged at {score:.1%} fraud probability — {level} risk. "
        f"Primary risk drivers are elevated '{factor1}' and '{factor2}' signals "
        f"relative to this cardholder's historical profile. "
        f"{actions[level]}"
    )


def generate_triage_narrative(transaction_row, model, explainer, feature_names, api_key=None):
    score, top_risk, top_protect = explain_transaction(
        transaction_row, model, explainer, feature_names
    )

    risk_factors = "\n".join([
        f"  - {r['feature']}: value={r['value']:.3f}, risk_contribution={r['shap']:.4f}"
        for _, r in top_risk.iterrows()
    ])
    protect_factors = "\n".join([
        f"  - {r['feature']}: value={r['value']:.3f}, contribution={r['shap']:.4f}"
        for _, r in top_protect.iterrows()
    ])

    prompt = f"""You are a financial crime risk analyst at a bank.
A transaction has been flagged by our fraud detection model with a score of {score:.1%}.

Top risk-increasing factors (from SHAP analysis):
{risk_factors}

Top risk-reducing factors:
{protect_factors}

Write a concise SAR-style triage narrative (3-4 sentences) that:
1. States the fraud probability and overall risk level (Low/Medium/High/Critical)
2. Explains the top 2-3 risk factors in plain English
3. Recommends an action: Auto-approve, Manual review, or Immediate block

Be specific and professional. Write for a compliance officer. No jargon."""

    key = api_key or os.environ.get('OPENROUTER_API_KEY')
    _MODELS = [
        ("deepseek/deepseek-chat:free",             "DeepSeek-Chat"),
        ("meta-llama/llama-3.3-70b-instruct:free",  "Llama-3.3-70B"),
        ("google/gemma-4-31b-it:free",              "Gemma-4-31B"),
        ("mistralai/mistral-7b-instruct:free",       "Mistral-7B"),
        ("qwen/qwen3-235b-a22b:free",               "Qwen3-235B"),
    ]

    narrative = source = None
    if key:
        import requests as _requests, time as _time
        for model_id, model_name in _MODELS:
            try:
                resp = _requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                    json={"model": model_id, "max_tokens": 300,
                          "messages": [{"role": "user", "content": prompt}]},
                    timeout=30,
                )
                if resp.status_code == 429:
                    _time.sleep(2)
                    continue
                resp.raise_for_status()
                content = resp.json().get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                if content:
                    narrative, source = content, model_name
                    break
            except Exception:
                continue

    if not narrative:
        narrative = _template_narrative(score, top_risk)
        source    = "Template (set OPENROUTER_API_KEY for LLM narratives)"

    return {
        'fraud_score':      round(score, 4),
        'risk_level':       _risk_level(score),
        'narrative':        narrative,
        'source':           source,
        'top_risk_factors': top_risk[['feature','shap']].to_dict('records')
    }

## Triage Demo — 5 Highest-Risk Transactions

Taking the 5 transactions from the validation set with the highest fraud scores and generating a case narrative for each.

In [ ]:
split_idx  = int(len(X_train) * 0.8)
X_val      = X_train.iloc[split_idx:]
y_val      = y_train.iloc[split_idx:]
val_scores = xgb_model.predict_proba(X_val)[:, 1]
top5_idx   = np.argsort(val_scores)[-5:][::-1]

triage_results = []
for rank, idx in enumerate(top5_idx, 1):
    result       = generate_triage_narrative(X_val.iloc[idx].values, xgb_model, explainer, FEATURES)
    actual_label = y_val.iloc[idx]

    print(f"\n{'─'*60}")
    print(f"  Case #{rank}  |  Actual: {'FRAUD ✓' if actual_label else 'LEGIT'}")
    print(f"  Score      : {result['fraud_score']:.1%}")
    print(f"  Risk level : {result['risk_level']}")
    print(f"  Source     : {result['source']}")
    print(f"\n  {result['narrative']}")
    print(f"\n  Top risk factors:")
    for f in result['top_risk_factors']:
        print(f"    • {f['feature']}: SHAP = {f['shap']:.4f}")

    triage_results.append({'rank': rank, 'actual_fraud': int(actual_label), **result})

triage_df = pd.DataFrame(triage_results)
triage_df.to_csv(OUT_DIR + 'triage_demo.csv', index=False)
print(f"\nTriage output saved → {OUT_DIR}triage_demo.csv")

## Final Results Summary

In [ ]:
with open(OUT_DIR + 'metrics.json') as f:
    metrics = json.load(f)

psi_summary = {'train_vs_test': {'psi': psi_value, 'status': interpret_psi(psi_value)}}
with open(OUT_DIR + 'psi_results.json', 'w') as f:
    json.dump(psi_summary, f, indent=2)

print("Results")
print("=" * 50)
for model_name, m in metrics.items():
    label = 'Champion' if 'decision_tree' in model_name else 'Challenger'
    name  = 'Decision Tree' if 'decision_tree' in model_name else 'XGBoost'
    print(f"\n  {label}: {name}")
    print(f"    KS Statistic : {m['ks']:.4f}")
    print(f"    Gini         : {m['gini']:.4f}")
    print(f"    AUC-ROC      : {m['auc_roc']:.4f}")
    print(f"    AUC-PR       : {m['auc_pr']:.4f}")

print(f"\n  PSI (train vs test) : {psi_value:.5f} — {interpret_psi(psi_value)}")
print(f"\n  All outputs → {OUT_DIR}")